# Text Preprocessing and Tokenization

Raw text from the web, social media, or documents is messy. It contains HTML tags, URLs, inconsistent capitalization, punctuation, contractions, emojis, and encoding artifacts. Before any NLP model can process text, it must be **cleaned**, **normalized**, and **tokenized** — split into the atomic units the model understands.

Preprocessing is not a one-size-fits-all step. A sentiment analysis system may lowercase text and remove stop words. A named entity recognizer must preserve capitalization to detect *"Apple"* the company. A machine translation system needs a tokenizer aligned with its vocabulary. The choices you make here directly affect downstream performance.

This notebook covers text cleaning, normalization, word and subword tokenization, stop word handling, stemming and lemmatization, vocabulary construction, and how to assemble these steps into a reusable preprocessing pipeline.

In [ ]:
import re
import unicodedata
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. Why Preprocessing Matters

Consider this raw text from a product review:

> *"OMG!! This product is AMAZING!!! 😍 Check it out at https://example.com — 10/10 would buy again!!!"*

A model that treats every character string literally sees *"AMAZING!!!"* and *"amazing"* as different tokens, wastes capacity on URLs and emojis, and may split *"10/10"* unpredictably. Preprocessing converts messy human input into a consistent, model-ready format.

**Goals of preprocessing:**
- Remove noise that carries no semantic signal (HTML, URLs, excess punctuation).
- Normalize surface variations (*"Running"*, *"running"*, *"RUNNING"* → consistent form).
- Split text into tokens the model can index and process.
- Reduce vocabulary size by collapsing word forms (stemming/lemmatization).

**The golden rule:** Preprocess training and inference data identically. A model trained on lowercased text will fail on mixed-case input at deployment if you forget to lowercase.

---

## 2. Text Cleaning

Cleaning removes structural noise from raw text. Common operations:

| Operation | Example Input | Example Output |
|-----------|--------------|----------------|
| Remove HTML tags | `<p>Hello</p>` | `Hello` |
| Remove URLs | `Visit https://x.com` | `Visit` |
| Remove email addresses | `Contact a@b.com` | `Contact` |
| Remove extra whitespace | `Hello   world` | `Hello world` |
| Remove digits | `Room 42` | `Room` |
| Remove special characters | `Hello!!!` | `Hello` |

Which operations to apply depends on the task. For topic classification, removing numbers may be fine. For information extraction of dates and prices, removing digits would be destructive.

Regular expressions (`re` module) are the workhorse for pattern-based cleaning in Python.

In [ ]:
def clean_text(text, remove_urls=True, remove_html=True, remove_digits=False,
               remove_extra_spaces=True):
    if remove_html:
        text = re.sub(r"<[^>]+>", " ", text)
    if remove_urls:
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"\S+@\S+\.\S+", " ", text)
    if remove_digits:
        text = re.sub(r"\d+", " ", text)
    # Remove emojis and non-ASCII symbols (keep basic Latin letters)
    text = re.sub(r"[^\w\s.,!?';-]", " ", text, flags=re.UNICODE)
    if remove_extra_spaces:
        text = re.sub(r"\s+", " ", text).strip()
    return text

raw_samples = [
    "<p>Check out https://example.com for details!</p>",
    "Contact us at support@company.com ASAP!!!",
    "OMG this is AMAZING 😍😍 10/10 would recommend!!!",
]

print(f"{'Raw':<55s} {'Cleaned'}")
print("-" * 90)
for raw in raw_samples:
    cleaned = clean_text(raw)
    print(f"{raw[:53]:<55s} {cleaned}")

---

## 3. Normalization

Normalization standardizes text to reduce superficial variation.

**Lowercasing** — *"Apple"* and *"apple"* become the same token. Essential for most classification tasks. Skip for NER and proper noun detection.

**Unicode normalization** — The character *"é"* can be stored as a single code point (NFC) or as *"e"* + combining accent (NFD). `unicodedata.normalize('NFKC', text)` converts to a canonical form, preventing duplicate vocabulary entries.

**Expanding contractions** — *"don't"* → *"do not"*, *"it's"* → *"it is"*. Helps models that tokenize by whitespace. Modern subword tokenizers often handle contractions without expansion.

**Accent removal** — Optional: *"café"* → *"cafe"*. Useful for search and matching across encodings.

In [ ]:
CONTRACTIONS = {
    "don't": "do not", "won't": "will not", "can't": "cannot",
    "it's": "it is", "i'm": "i am", "you're": "you are",
    "they've": "they have", "we'll": "we will", "isn't": "is not",
}

def normalize_text(text, lowercase=True, expand_contractions=True):
    text = unicodedata.normalize("NFKC", text)
    if lowercase:
        text = text.lower()
    if expand_contractions:
        for contraction, expansion in CONTRACTIONS.items():
            text = re.sub(r"\b" + re.escape(contraction) + r"\b", expansion, text)
    return text

samples = [
    "I don't think it's a bad idea.",
    "Café résumé naïve",  # unicode accents
    "They've said we'll know soon.",
]

for s in samples:
    print(f"Original:  {s}")
    print(f"Normalized: {normalize_text(s)}\n")

---

## 4. Tokenization

**Tokenization** splits text into tokens — the atomic units processed by a model.

| Tokenization Level | Token Unit | Use Case |
|-------------------|-----------|----------|
| **Word** | Whole words | Classical NLP, bag-of-words |
| **Subword** | Word pieces (BPE, WordPiece) | Modern LLMs, handles OOV |
| **Character** | Individual characters | Morphologically rich languages, very small vocab |
| **Sentence** | Full sentences | Document-level processing |

Word tokenization sounds simple — split on spaces — but edge cases abound:
- *"don't"* — one token or two (*"do"*, *"n't"*)?
- *"U.S.A."* — one token or four?
- *"state-of-the-art"* — hyphenated compounds.
- *"$42.50"* — numbers with punctuation.

Production systems use carefully designed tokenizers (spaCy, Hugging Face tokenizers) rather than naive `split()`.

In [ ]:
def word_tokenize(text):
    """Regex-based word tokenizer preserving contractions as single tokens."""
    pattern = r"\b[\w']+\b"
    return re.findall(pattern, text.lower())

def sentence_tokenize(text):
    """Split text into sentences using punctuation boundaries."""
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s.strip() for s in sentences if s.strip()]

text = "Dr. Smith arrived at 3 p.m. He didn't wait. The U.S.A. team won!"
print("Word tokens:", word_tokenize(text))
print()
print("Sentences:")
for i, sent in enumerate(sentence_tokenize(text), 1):
    print(f"  {i}. {sent}")

---

## 5. Stop Words

**Stop words** are extremely common words that carry little discriminative meaning: *"the"*, *"is"*, *"at"*, *"which"*, *"on"*. Removing them reduces vocabulary size and noise for tasks like topic modeling and search indexing.

**When to remove stop words:**
- Bag-of-words classification and clustering.
- Information retrieval where content words matter most.

**When to keep stop words:**
- Machine translation and text generation (grammar depends on them).
- Sentiment analysis where negation matters (*"not good"*).
- Named entity recognition and parsing.
- Neural models and LLMs (they learn stop word usage from context).

Modern deep learning models typically **do not** remove stop words — the model learns their role. Classical ML pipelines often do.

In [ ]:
STOP_WORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "to", "of", "in", "for",
    "on", "with", "at", "by", "from", "as", "into", "through", "during",
    "it", "its", "this", "that", "these", "those", "i", "you", "he",
    "she", "we", "they", "what", "which", "who", "and", "but", "or",
    "not", "no", "so", "if", "then", "than", "too", "very", "just",
}

def remove_stop_words(tokens, stop_words=STOP_WORDS):
    return [t for t in tokens if t.lower() not in stop_words]

sentence = "The machine learning model is not bad, but it is not great either."
tokens = word_tokenize(sentence)
filtered = remove_stop_words(tokens)

print(f"Original ({len(tokens)} tokens): {tokens}")
print(f"Filtered ({len(filtered)} tokens): {filtered}")
print("\nNote: removing 'not' would lose negation — a reason to keep stop words for sentiment.")

---

## 6. Stemming

**Stemming** reduces words to their root form by stripping suffixes heuristically. It is fast but imprecise.

| Word | Stem (Porter) |
|------|--------------|
| running | run |
| studies | studi |
| happily | happili |
| universe | univers |

The **Porter stemmer** (1980) applies a sequence of suffix rules: remove *-ing*, *-ed*, *-ly*, *-ness*, etc. **Snowball stemmers** extend Porter to other languages.

Stemming does not guarantee valid words (*"studies"* → *"studi"*). It does not use vocabulary or context — purely rule-based chopping.

In [ ]:
def simple_stem(word):
    """Rule-based stemmer covering common English suffixes."""
    word = word.lower()
    suffixes = [
        ("ization", "ize"), ("ational", "ate"), ("fulness", "ful"),
        ("ousness", "ous"), ("iveness", "ive"), ("biliti", "ble"),
        ("ation", "ate"), ("ator", "ate"), ("alli", "al"), ("fulness", "ful"),
        ("ousli", "ous"), ("iviti", "ive"), ("able", ""), ("ible", ""),
        ("ment", ""), ("ness", ""), ("ingly", ""), ("edly", ""),
        ("ing", ""), ("edly", ""), ("edly", ""), ("ed", ""),
        ("ly", ""), ("es", ""), ("s", ""),
    ]
    for suffix, replacement in suffixes:
        if word.endswith(suffix) and len(word) - len(suffix) + len(replacement) >= 3:
            return word[: -len(suffix)] + replacement
    return word

words = ["running", "runs", "runner", "studies", "studying", "university",
         "universities", "happily", "happiness", "connection", "connected"]

print(f"{'Word':<15s} {'Stem':<15s}")
print("-" * 30)
for w in words:
    print(f"{w:<15s} {simple_stem(w):<15s}")

---

## 7. Lemmatization

**Lemmatization** reduces words to their dictionary form (**lemma**) using vocabulary and morphological analysis. Unlike stemming, lemmatization produces valid words.

| Word | Lemma |
|------|-------|
| running | run |
| studies | study |
| better | good |
| mice | mouse |

Lemmatization requires **part-of-speech** information. *"running"* as a verb lemmatizes to *"run"*; as a noun, it stays *"running"* (a type of activity). Stemmers ignore POS; lemmatizers use it.

Lemmatization is slower but more accurate than stemming. Use stemming for speed-critical retrieval; lemmatization when word quality matters.

In [ ]:
# Simplified lemmatization lookup (production systems use morphological analyzers)
LEMMA_MAP = {
    "running": "run", "runs": "run", "ran": "run", "runner": "runner",
    "studies": "study", "studying": "study", "studied": "study",
    "universities": "university", "university": "university",
    "happily": "happy", "happiness": "happy", "happier": "happy",
    "connections": "connection", "connected": "connect", "connecting": "connect",
    "mice": "mouse", "better": "good", "went": "go", "was": "be",
    "children": "child", "feet": "foot", "geese": "goose",
}

def simple_lemmatize(word):
    return LEMMA_MAP.get(word.lower(), word.lower())

print(f"{'Word':<15s} {'Stem':<15s} {'Lemma':<15s}")
print("-" * 45)
for w in words:
    print(f"{w:<15s} {simple_stem(w):<15s} {simple_lemmatize(w):<15s}")

---

## 8. N-grams

An **n-gram** is a contiguous sequence of $n$ tokens. N-grams capture local word order that bag-of-words loses.

- **Unigrams** ($n=1$): individual words — `["machine", "learning", "is"]`
- **Bigrams** ($n=2$): word pairs — `["machine learning", "learning is"]`
- **Trigrams** ($n=3$): triples — `["machine learning is"]`

N-grams improve context sensitivity. *"not good"* as a bigram carries different sentiment than *"good"* alone. The tradeoff: vocabulary explodes. With vocabulary size $V$, there are up to $V^2$ bigrams and $V^3$ trigrams.

In [ ]:
def generate_ngrams(tokens, n):
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

tokens = word_tokenize("machine learning is transforming natural language processing")

for n in [1, 2, 3]:
    ngrams = generate_ngrams(tokens, n)
    label = {1: "Unigrams", 2: "Bigrams", 3: "Trigrams"}[n]
    print(f"{label}: {ngrams}")

---

## 9. Building a Vocabulary

A **vocabulary** maps tokens to integer indices. Steps:

1. Tokenize all training documents.
2. Count token frequencies.
3. Keep tokens above a minimum frequency threshold (removes rare typos).
4. Optionally cap vocabulary size (keep most frequent $N$ tokens).
5. Reserve special tokens for unknown and padding.

| Special Token | Purpose |
|--------------|--------|
| `<PAD>` | Fill shorter sequences to uniform length |
| `<UNK>` | Represent out-of-vocabulary tokens |
| `<SOS>` / `<EOS>` | Mark sequence start and end |
| `<MASK>` | Masked language model training |

The vocabulary is built **only from training data**. Including test data leaks information and inflates vocabulary with tokens the model should treat as unknown.

In [ ]:
class Vocabulary:
    def __init__(self, min_freq=1, max_size=None):
        self.min_freq = min_freq
        self.max_size = max_size
        self.token2idx = {"<PAD>": 0, "<UNK>": 1}
        self.idx2token = {0: "<PAD>", 1: "<UNK>"}

    def build(self, documents):
        counter = Counter()
        for doc in documents:
            counter.update(word_tokenize(doc))
        tokens = [t for t, c in counter.most_common() if c >= self.min_freq]
        if self.max_size:
            tokens = tokens[: self.max_size]
        for token in tokens:
            if token not in self.token2idx:
                idx = len(self.token2idx)
                self.token2idx[token] = idx
                self.idx2token[idx] = token
        return self

    def encode(self, text):
        return [self.token2idx.get(t, 1) for t in word_tokenize(text)]

    def decode(self, indices):
        return [self.idx2token.get(i, "<UNK>") for i in indices]

    def __len__(self):
        return len(self.token2idx)

corpus = [
    "machine learning models learn from data",
    "deep learning uses neural networks",
    "natural language processing handles text data",
    "neural networks learn hierarchical features",
]

vocab = Vocabulary(min_freq=1).build(corpus)
print(f"Vocabulary size: {len(vocab)}")
print(f"First 10 entries: {list(vocab.token2idx.items())[:10]}")

encoded = vocab.encode("neural networks process language")
print(f"\nEncode: 'neural networks process language'")
print(f"  Indices: {encoded}")
print(f"  Tokens:  {vocab.decode(encoded)}")

---

## 10. Padding and Truncation

Neural models expect fixed-length inputs (or batched sequences of equal length). **Padding** adds `<PAD>` tokens to shorter sequences. **Truncation** cuts longer sequences to a maximum length.

```
Sequence 1: [12, 45, 7]           → [12, 45, 7, 0, 0]   (pad to length 5)
Sequence 2: [3, 88, 21, 9, 55, 1] → [3, 88, 21, 9, 55]  (truncate to length 5)
```

When using RNNs, pack padded sequences to skip pad tokens during computation. Attention-based models use **attention masks** to ignore padded positions.

In [ ]:
def pad_sequences(sequences, max_len, pad_value=0):
    padded = []
    for seq in sequences:
        if len(seq) >= max_len:
            padded.append(seq[:max_len])
        else:
            padded.append(seq + [pad_value] * (max_len - len(seq)))
    return np.array(padded)

seqs = [
    vocab.encode("machine learning"),
    vocab.encode("deep neural networks learn features"),
    vocab.encode("text"),
]

print("Original sequences:")
for s in seqs:
    print(f"  {s} (len={len(s)})")

padded = pad_sequences(seqs, max_len=6)
print(f"\nPadded (max_len=6):\n{padded}")

---

## 11. Subword Tokenization

Word-level tokenization fails on **out-of-vocabulary (OOV)** words — any word not seen during training maps to `<UNK>`, losing information. Subword tokenization splits rare words into known pieces.

*"unhappiness"* → `["un", "##happiness"]` or `["unhappy", "##ness"]`

Every word can be represented from a finite subword vocabulary. Common algorithms:

**Byte Pair Encoding (BPE)** — Start with character vocabulary. Iteratively merge the most frequent adjacent pair until vocabulary reaches target size. Used by GPT models.

**WordPiece** — Similar to BPE but merges pairs that maximize language model likelihood. Used by BERT.

**SentencePiece** — Treats text as a raw byte stream (no pre-tokenization). Used by T5, Llama.

Subword tokenization is the standard for modern language models. GPT-4's tokenizer has ~100,000 subword units.

In [ ]:
def train_bpe(corpus, num_merges):
    """Simplified BPE: learn merge rules from a corpus."""
    # Initialize: each word as space-separated characters + end marker
    word_freqs = Counter()
    for word in corpus:
        word_freqs[" ".join(list(word)) + " </w>"] += 1

    merges = []
    for _ in range(num_merges):
        pair_counts = Counter()
        for word, freq in word_freqs.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pair_counts[(symbols[i], symbols[i+1])] += freq
        if not pair_counts:
            break
        best_pair = pair_counts.most_common(1)[0][0]
        merges.append(best_pair)
        a, b = best_pair
        new_token = a + b
        new_word_freqs = Counter()
        for word, freq in word_freqs.items():
            new_word = word.replace(f"{a} {b}", new_token)
            new_word_freqs[new_word] = freq
        word_freqs = new_word_freqs

    return merges, word_freqs

bpe_corpus = ["low"] * 5 + ["lower"] * 2 + ["newest"] * 6 + ["widest"] * 3
merges, final_vocab = train_bpe(bpe_corpus, num_merges=10)

print("BPE merge rules learned:")
for i, (a, b) in enumerate(merges, 1):
    print(f"  {i:2d}. '{a}' + '{b}' → '{a}{b}'")

print("\nFinal subword representations:")
for word_repr, freq in sorted(final_vocab.items(), key=lambda x: -x[1]):
    print(f"  {word_repr:20s} (freq={freq})")

---

## 12. Character-Level Tokenization

At the finest granularity, each character is a token. Vocabulary is tiny (~100 characters for English including punctuation) and there are **no OOV tokens** — any string can be represented.

**Advantages:** Handles any spelling, morphology, and novel words. Useful for languages with rich morphology (Turkish, Finnish) and for tasks like spell correction.

**Disadvantages:** Sequences are much longer. The model must learn to compose characters into words before learning semantics. Training is slower.

Character-level models are less common today but remain useful for specialized applications and as a teaching tool for understanding sequence modeling.

In [ ]:
def char_tokenize(text):
    return list(text)

word = "hello"
print(f"Word-level:  {word_tokenize(word)}")
print(f"Char-level:  {char_tokenize(word)}")
print(f"\nWord 'unhappiness' → {len(word_tokenize('unhappiness'))} word token")
print(f"Word 'unhappiness' → {len(char_tokenize('unhappiness'))} char tokens")

---

## 13. Out-of-Vocabulary Handling

When a test-time token is not in the vocabulary, strategies include:

| Strategy | How It Works |
|----------|-------------|
| **`<UNK>` token** | Map all unknown words to a single index |
| **Subword tokenization** | Split unknown word into known subwords |
| **Character fallback** | Represent unknown word character by character |
| **Hashing trick** | Hash word to a fixed bucket (collisions possible) |

Subword tokenization largely solved the OOV problem for modern NLP. A word like *"cryptocurrency"* may not be in training data, but subwords *"crypto"* and *"##currency"* likely are.

---

## 14. End-to-End Preprocessing Pipeline

Combine all steps into a reusable pipeline with configurable options:

In [ ]:
class TextPreprocessor:
    def __init__(self, lowercase=True, remove_stop=True, stem=False,
                 min_freq=1, max_vocab=None):
        self.lowercase = lowercase
        self.remove_stop = remove_stop
        self.stem = stem
        self.vocab = Vocabulary(min_freq=min_freq, max_size=max_vocab)
        self._fitted = False

    def _process_tokens(self, text):
        text = clean_text(text)
        text = normalize_text(text, lowercase=self.lowercase)
        tokens = word_tokenize(text)
        if self.remove_stop:
            tokens = remove_stop_words(tokens)
        if self.stem:
            tokens = [simple_stem(t) for t in tokens]
        return tokens

    def fit(self, documents):
        processed = [" ".join(self._process_tokens(doc)) for doc in documents]
        self.vocab.build(processed)
        self._fitted = True
        return self

    def transform(self, text):
        tokens = self._process_tokens(text)
        return [self.vocab.token2idx.get(t, 1) for t in tokens]

    def fit_transform(self, documents):
        return self.fit(documents).transform_documents(documents)

    def transform_documents(self, documents):
        return [self.transform(doc) for doc in documents]

reviews = [
    "This product is AMAZING! Best purchase ever.",
    "Terrible quality. Do NOT buy this item.",
    "It's okay, nothing special but works fine.",
    "The BEST product I've ever used!!! https://shop.com",
]

preprocessor = TextPreprocessor(lowercase=True, remove_stop=True, stem=True)
encoded_docs = preprocessor.fit_transform(reviews)

print(f"Vocabulary size: {len(preprocessor.vocab)}\n")
for review, encoded in zip(reviews, encoded_docs):
    tokens = preprocessor.vocab.decode(encoded)
    print(f"Original: {review[:50]}...")
    print(f"Tokens:   {tokens}")
    print(f"Indices:  {encoded}\n")

---

## 15. Preprocessing Impact on Vocabulary

Different preprocessing choices dramatically affect vocabulary size and token distributions.

In [ ]:
large_corpus = [
    "The researchers are studying machine learning algorithms.",
    "Machine learning researchers study algorithms and models.",
    "Deep learning models are studied by researchers worldwide.",
    "Algorithms for machine learning are studied extensively.",
    "Researchers study deep learning and machine learning models.",
] * 10

configs = [
    ("Raw (no processing)", {"lowercase": False, "remove_stop": False, "stem": False}),
    ("Lowercase only", {"lowercase": True, "remove_stop": False, "stem": False}),
    ("+ Stop words removed", {"lowercase": True, "remove_stop": True, "stem": False}),
    ("+ Stemming", {"lowercase": True, "remove_stop": True, "stem": True}),
]

vocab_sizes = []
labels = []
for label, kwargs in configs:
    pp = TextPreprocessor(**kwargs).fit(large_corpus)
    vocab_sizes.append(len(pp.vocab))
    labels.append(label)

plt.figure(figsize=(8, 4))
bars = plt.barh(labels, vocab_sizes, color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
plt.xlabel("Vocabulary size")
plt.title("Preprocessing reduces vocabulary size")
for bar, size in zip(bars, vocab_sizes):
    plt.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             str(size), va="center")
plt.tight_layout()
plt.show()

---

## 16. Best Practices

**Match preprocessing to the task.** NER needs capitalization; sentiment may not. Translation must keep all words.

**Fit on training data only.** Vocabulary, IDF statistics, and BPE merges must come from training set — never validation or test.

**Save and version your preprocessor.** Deployment must apply the exact same pipeline used during training.

**Don't over-preprocess for neural models.** Modern transformers use subword tokenizers and learn from raw-ish text. Aggressive stemming hurts more than helps.

**Handle edge cases explicitly.** Empty strings, single-character inputs, and documents in unexpected languages should have defined behavior.

**Use established tokenizers for LLMs.** Never build your own BPE for GPT or BERT — use the official tokenizer that matches the pre-trained vocabulary.

---

## 17. Summary

Text preprocessing transforms raw, messy language into a consistent format models can process. **Cleaning** removes HTML, URLs, and noise. **Normalization** handles case, unicode, and contractions. **Tokenization** splits text into words, subwords, or characters.

**Stop word removal** and **stemming/lemmatization** reduce vocabulary size for classical ML pipelines. **Vocabulary construction** maps tokens to indices with special tokens for padding and unknown words. **Subword tokenization** (BPE, WordPiece) is the standard for modern language models, solving the out-of-vocabulary problem.

The preprocessing pipeline must be consistent between training and inference, fit only on training data, and tailored to the specific NLP task. Good preprocessing does not guarantee good models — but bad preprocessing guarantees bad ones.